# WP43 — Tangled Hierarchy Detector

**Prometheus v0 PoC · Work Package 43**

> *"A strange loop occurs whenever, by moving only upwards (or only downwards) through the levels of some hierarchical system, we unexpectedly find ourselves back where we started."*
> — Douglas Hofstadter, *Gödel, Escher, Bach*, 1979

## What WP43 does

WP41 (Strange Loop Visualiser) records every cross-level signal and detects loops qualitatively.  WP43 goes further: it quantifies the **degree** of tangling with a single interpretable scalar — the **TanglingScore** — and compares it against a null (purely nested) baseline.

### TanglingScore
$$\text{TanglingScore} = \frac{2 \cdot \min(\text{upper\_mass},\, \text{lower\_mass})}{\text{upper\_mass} + \text{lower\_mass} + \varepsilon}$$

- **Score = 0** → purely nested (signals flow one way only)
- **Score = 1** → perfectly tangled (equal flow in both directions)

### Null baseline
Generation indices are shuffled, destroying causal ordering while preserving signal volumes.  The excess above the null baseline is the *genuine* causal tangling score.

### Classification
| Score | Excess | Label |
|-------|--------|-------|
| < 0.15 | any | NESTED |
| 0.15–0.50 | any | PARTIALLY_TANGLED |
| 0.50–0.80 | any | TANGLED |
| ≥ 0.80 | ≥ 0.20 | STRANGE_LOOP |


In [ ]:
import sys, os
# Allow running from the repo root or from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import logging
logging.basicConfig(level=logging.WARNING)

from prometheus.wp43_tangled_hierarchy import (
    run_tangling_demo,
    verify_wp43_exit_criteria,
    TanglingAnalyser,
    HierarchyClassification,
)
print("WP43 loaded ✓")


## Run the Tangling Analysis

Simulate 150 generations of the three-level CRLS hierarchy and compute the TanglingScore.

In [ ]:
report = run_tangling_demo(n_generations=150, seed=42)
print(report.summary())


## TanglingScore Time-Series

Does tangling *grow* across generations as Good's hypothesis predicts?

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use("Agg")
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

if HAS_MPL and report.time_series:
    starts = [w.window_start for w in report.time_series]
    scores = [w.score        for w in report.time_series]
    loops  = [w.n_loops      for w in report.time_series]

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    ax = axes[0]
    ax.plot(starts, scores, "b-o", markersize=4, label="TanglingScore")
    ax.axhline(report.null_score, color="gray", linestyle="--", label=f"Null baseline ({report.null_score:.3f})")
    ax.axhline(0.50, color="orange", linestyle=":", alpha=0.6, label="TANGLED threshold")
    ax.axhline(0.80, color="red",    linestyle=":", alpha=0.6, label="STRANGE_LOOP threshold")
    ax.fill_between(starts, report.null_score, scores, alpha=0.15, color="blue", label="Excess tangling")
    ax.set_ylabel("TanglingScore")
    ax.set_title(f"TanglingScore over time  |  Classification: {report.classification.value}  |  Trend: {report.tangling_trend}")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    ax2.bar(starts, loops, width=4, color="steelblue", alpha=0.7, label="Bidirectional loops")
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Detected loops")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("wp43_tangling_timeseries.png", dpi=100)
    plt.show()
    print("Plot saved → wp43_tangling_timeseries.png")
else:
    print("matplotlib not available — numeric results only")
    for w in report.time_series[:6]:
        print(f"  gen {w.window_start:3d}–{w.window_end:3d}  score={w.score:.4f}  loops={w.n_loops}")


## Per-Level-Pair Bidirectional Coupling

Which level pairs are most tightly coupled?

In [ ]:
print("Bidirectional coupling strengths (geometric mean of E[A→B] × E[B→A]):")
print()
for pair, strength in sorted(report.per_pair_coupling.items(), key=lambda x: -x[1]):
    bar = "█" * int(strength * 300)
    print(f"  {pair:35s}  {strength:.5f}  {bar}")
print()
print(f"Observed TanglingScore : {report.score:.4f}")
print(f"Null baseline score    : {report.null_score:.4f}")
print(f"Excess (genuine)       : {report.excess:.4f}")
print(f"Classification         : {report.classification.value}")
print(f"Detected loops         : {report.n_loops}")
print(f"Tangling trend         : {report.tangling_trend}")


## Hofstadter Verdict

In [ ]:
print(report.hofstadter_verdict)


## WP43 Exit Criteria

In [ ]:
criteria = verify_wp43_exit_criteria(report)
all_pass = all(criteria.values())
print(f"{'PASS' if all_pass else 'FAIL'} — WP43 Exit Criteria")
print()
for name, result in criteria.items():
    status = "✓" if result else "✗"
    print(f"  [{status}] {name}")
print()
print(f"All criteria pass: {all_pass}")


## Conclusion

WP43 makes Hofstadter's qualitative notion of *tangled vs. nested* hierarchies **quantitative and falsifiable**:

1. **TanglingScore** is a single scalar in [0, 1] derived from the entanglement matrix, comparing upward and downward inter-level flow.
2. **Null baseline** (shuffled generation indices) confirms that the observed tangling is causal, not coincidental.
3. **Time series** shows how tangling evolves — and whether it grows as the system learns (consistent with Good's intelligence explosion hypothesis).
4. **Classification** maps the score to Hofstadter's taxonomy: NESTED → PARTIALLY_TANGLED → TANGLED → STRANGE_LOOP.

**Theoretical bridge**: WP43's TanglingScore is analogous to Tononi's Φ (integrated information) at the *architectural* level.  Both measure the degree to which a system exceeds the sum of its independent parts.
